# World Models for Physical AI: Planning & Distillation at Scale on Anyscale

### Search inside a robot's **world model** in the cloud with Ray, then distill it into a fast on-robot policy

Today's robot policies (VLAs) are amazing **reflexes** (e.g. image + instruction → action, in one shot). But a reflex can't *deliberate*. It can't imagine three ways a grasp might fail and pick the one that won't. The missing piece is a **world model**. This is a learned model of *what happens next, given an action*, that the robot can **imagine and search inside** to choose what to do. (LLMs crossed a similar line with **test-time compute**. This is the "think longer, answer better" unlock behind o1-style reasoning.)

But that search is the expensive part. A reflex is a single forward pass. Choosing an action by deliberation means rolling a billion-parameter model forward over many imagined futures and optimizing across them, and the better you want the action, the more futures you have to search. The action space is high-dimensional and continuous, so you can't enumerate it, and every candidate you score is another full rollout of the model. That is what makes action selection so much heavier than a reflex, and it is why this work belongs in the cloud.

**Autonomous vehicles (AV) already proves the architecture** where the expensive compute stays off-vehicle and elastic in the cloud, while a tightly-budgeted policy runs on the car. In AV that off-vehicle compute is large-scale *simulation* — historically hand-built, increasingly learned generative neural simulators like Wayve's GAIA. This notebook generalizes the same split to *general-purpose* robots, where you can't hand-author a simulator for every object and task. We do this by swapping in an open, learned **world model** the robot can *search inside* to plan, and shows the production architecture for **world-model inference** on Ray + Anyscale:

> Take an open **world model**, **imagine and search inside it** to choose actions, run that search at scale in the cloud, and **distill** the result into a fast policy that runs on the robot.

**A fast brain on the robot. A slow brain — the world model — runs in the cloud.**

## This is the autonomous-driving playbook

The slow/fast split is already how self-driving works, the most mature robotics domain in production:

- **The heavy thinking runs off-vehicle:** AV programs lean on **large-scale simulation** in the cloud. They replay and synthesize driving scenarios to plan, test, and train against, before anything touches the road.
- **Search against a predictive model *is* the planner.** **Model-predictive control:** Roll candidate trajectories forward through a predictive model, score them, and keep the best. This is a standard AV planning method. That's the same search loop we run here.
- **A distilled, tightly-budgeted policy runs on the car:** The onboard driving policy executes inside a hard real-time latency budget, while the expensive simulation, search, and evaluation stay in the cloud.

**The catch** is that AV leans on **hand-built simulators**. That works because driving lives in a relatively structured world — roads, lanes, vehicles — but you **can't hand-author a simulator for every object and task** a general-purpose robot will face. The unlock for arms, humanoids, and mobile manipulators is to replace the hand-built simulator with a **learned, general world model** you can search inside. **V-JEPA 2-AC** is one such open model. This template wires it into the same **cloud-search → distill → deploy** loop the AV industry already trusts, all orchestrated on Ray + Anyscale.

*(Learned/generative world models are starting to appear in AV too (e.g. Wayve's **GAIA**) but production planning still runs largely on hand-built simulation. This template is about making the **learned** world model the substrate.)*

One idea is worth pausing on, because the rest of the notebook leans on it. A model like V-JEPA does not try to predict the next camera image pixel by pixel. It first compresses each observation into a compact vector of features, called a *latent*, and learns to predict how that latent changes when the robot acts. Most of what fills a raw image, like the exact texture of the table or the lighting, has nothing to do with choosing a good action, and predicting all of it is both wasteful and challenging. Working in latent space keeps only what matters for *what happens next*, which is what makes it practical to imagine and compare thousands of candidate futures. So when we say the robot imagines and searches, it is doing that in this compact latent space, not by rendering video.

## The bet

A world model predicts *what happens next* given an action. Once you have one, action selection becomes **search**: sample candidate action sequences, roll them forward in the model's imagination, and keep the one whose imagined outcome is closest to the goal. More compute → more candidates and longer lookahead → better actions. That is **test-time compute for control**.

Yann LeCun's team has bet heavily on this direction with **V-JEPA**, a self-supervised video world model that predicts in *latent* space rather than pixels. We use the open, action-conditioned **V-JEPA 2-AC** (MIT licensed) as our substrate.

This is the workload Ray + Anyscale exist to run: heavy, parallel search that scales horizontally across the cluster, with only the distilled policy shipped to the edge.

## The architecture

![Slow brain in the cloud, fast brain on the robot](architecture.png)

The slow brain thinks continuously in the cloud, and the fleet pulls distilled updates. Let's build it.

---
## 1. Connect to the cluster

We ship our slow-brain module (`slow_brain.py`) to every node with `py_modules`, and point the world-model weights cache at shared cluster storage so each GPU loads the 11 GB checkpoint locally instead of re-downloading it.

In [ ]:
import os, sys, time, numpy as np, ray

REPO_DIR = "/home/ray/default/robotics_world_model_inference"
sys.path.insert(0, REPO_DIR)   # so the driver (this notebook) can import slow_brain too
TORCH_HOME = "/mnt/cluster_storage/torch_hub"   # shared cache: weights + V-JEPA repo
VJEPA_REPO = f"{TORCH_HOME}/hub/facebookresearch_vjepa2_main"

ray.init(address="auto", logging_level="ERROR", runtime_env={
    "py_modules": [f"{REPO_DIR}/slow_brain.py"],
    "env_vars": {"TORCH_HOME": TORCH_HOME, "HF_HOME": "/mnt/cluster_storage/hf_cache"},
})

n_gpus = int(ray.cluster_resources().get("GPU", 0))
print(f"Cluster GPUs available for slow-brain planners: {n_gpus}")

## 2. A task for the world model

We use a short Franka manipulation clip shipped with V-JEPA 2: a **context** frame (where the arm is now) and a **goal** frame (where we want it), plus the robot's proprioceptive **state** (7-D end-effector pose). The slow brain's job: find the action that moves the imagined latent from context toward goal.

In [ ]:
import matplotlib.pyplot as plt

traj = np.load(f"{VJEPA_REPO}/notebooks/franka_example_traj.npz")
ctx_img, goal_img = traj["observations"][0, 0], traj["observations"][0, 1]
pose = traj["states"][0, 0]   # 7-DoF end-effector pose

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(ctx_img); ax[0].set_title("context (now)"); ax[0].axis("off")
ax[1].imshow(goal_img); ax[1].set_title("goal"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("images:", ctx_img.shape, "| pose (7-D):", np.round(pose, 3))

## 3. The slow brain imagines and searches

We spin up one **`Planner`**. This is a Ray actor that loads V-JEPA 2-AC onto a GPU once and then serves planning calls. Inside, planning is the **Cross-Entropy Method (CEM)**. This works as follows: sample a batch of candidate actions, roll each one forward through the action-conditioned predictor *in latent space*, score how close the imagined result lands to the goal latent (lower **energy** = closer), keep the best, refit the sampling distribution, and repeat.

![How the slow brain plans: sample, imagine, score, refine](cem_planning.png)

Two knobs set how hard it "thinks": **`samples`** (candidates per step) and **`cem_steps`** (refinement iterations). That product is our test-time compute dial.

In [ ]:
from slow_brain import Planner

planner = Planner.remote()   # loads the 1.3B world model onto one GPU (~45s, one time)

res = ray.get(planner.plan_from_images.remote(
    ctx_img, goal_img, pose, samples=64, cem_steps=4, rollout=1))

print(f"planned action (7-D delta): {np.round(res['action'], 3)}")
print(f"energy to goal (lower=better): {res['energy']:.4f}")
print(f"planning time on one L4:       {res['seconds']:.2f} s  (light config)")

## 4. Why this can't run on the robot

A robot control loop runs at **10–50 ms per action**, and full-strength planning is nowhere close. The cell above used a deliberately *light* config so it returns in a couple of seconds. A **production-strength** search, hundreds of candidates and ten refinement steps, is far heavier. Our dev cluster runs **L4** GPUs, a modest inference card, and we measured both points directly:

| config | samples × cem_steps × rollout | latency / action (L4, measured) |
|---|---|---|
| light (above) | 64 × 4 × 1 | ~2.4 s |
| **production-strength** | **400 × 10 × 2** | **~98 s** |

Nobody serves this on an L4 in production. On the accelerators you would actually use, an H100 at a minimum and more likely a B200 or better, that ~98 s drops toward **single-digit seconds per action** (we quantify this in §8). But single-digit seconds is still tens to hundreds of times slower than a 10–50 ms control loop. Full-strength planning is too slow to run on-robot on *any* GPU, so the gap is structural rather than a hardware deficiency. That is the reason the slow brain lives in the cloud while a distilled policy runs on the robot. Below we make the compute↔quality trade-off explicit, then scale it with Ray.

---
## 5. Ray as the orchestration substrate

Planning scales along **two independent axes**, and Ray fans out both:

1. **Deeper**: spend more compute per action (bigger search) for a better action. This is the *o1 curve for robots*.
2. **Wider**: plan for many robots / many states at once. This is *fleet throughput*.

First, the **deeper** axis: sweep the compute dial and watch plan quality (energy) improve. We distribute the sweep across all available GPUs.

In [ ]:
# Reuse the planner from §3 and add one per *remaining* GPU, so we use exactly n_gpus GPUs.
# (Creating n_gpus NEW ones here would mean n_gpus+1 actors for n_gpus GPUs -> the extra deadlocks.)
planners = [planner] + [Planner.remote() for _ in range(max(0, n_gpus - 1))]
ray.get([p.plan_from_images.remote(ctx_img, goal_img, pose, samples=8, cem_steps=1)
         for p in planners])   # warmup / model load on every GPU
print(f"{len(planners)} slow-brain planners online (1 reused from §3).")

In [ ]:
# DEEPER axis: increasing test-time compute per action, fanned across GPUs.
compute_dial = [(32, 2), (64, 3), (128, 5), (256, 8)]   # (samples, cem_steps), rollout=1

futs = [planners[i % len(planners)].plan_from_images.remote(
            ctx_img, goal_img, pose, samples=s, cem_steps=c, rollout=1)
        for i, (s, c) in enumerate(compute_dial)]
sweep = ray.get(futs)

for (s, c), r in zip(compute_dial, sweep):
    print(f"samples={s:>3} cem_steps={c:>2}  (compute={s*c:>4})  "
          f"energy={r['energy']:.4f}  latency={r['seconds']:5.2f}s")

In [ ]:
# The 'o1 curve for robots': more test-time compute -> better plan (lower energy), at a latency cost.
compute = [s * c for s, c in compute_dial]
energy = [r["energy"] for r in sweep]
latency = [r["seconds"] for r in sweep]

fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.plot(compute, energy, "o-", color="C0", lw=2, label="plan energy (↓ better)")
ax1.set_xlabel("test-time compute per action  (samples × CEM steps)")
ax1.set_ylabel("energy to goal", color="C0"); ax1.set_xscale("log")
ax2 = ax1.twinx()
ax2.plot(compute, latency, "s--", color="C3", lw=2, label="latency (s)")
ax2.set_ylabel("latency / action (s)", color="C3")
ax1.set_title("Test-time compute for robots: think longer, act better")
fig.tight_layout(); plt.show()

**Read the curve.** Spending more compute inside the world model buys better actions. This is the same "think longer, answer better" law that powers o1-style reasoning, now for control. The cloud is where you can afford to ride this curve, while the robot is not. Ray lets you pick any point on it and run it at fleet scale.

## 6. The wider axis: fleet throughput

Now hold compute fixed and scale the **number of planners**. This is the real production question: *how many robot-states per second can the cloud plan for?* Ray scales it near-linearly with GPUs.

In [ ]:
# WIDER axis: fixed light config, measure throughput as we add planners (GPUs).
N_TASKS = 4 * len(planners)
cfg = dict(samples=64, cem_steps=4, rollout=1)

throughput = []
for k in range(1, len(planners) + 1):
    pool = planners[:k]
    t0 = time.time()
    ray.get([pool[i % k].plan_from_images.remote(ctx_img, goal_img, pose, **cfg)
             for i in range(N_TASKS)])
    dt = time.time() - t0
    throughput.append(N_TASKS / dt)
    print(f"{k} planner(s): {N_TASKS} actions in {dt:5.1f}s  ->  {N_TASKS/dt:.2f} actions/s")

gpus = list(range(1, len(planners) + 1))
ideal = [throughput[0] * k for k in gpus]   # perfect linear scaling from the 1-GPU baseline

plt.figure(figsize=(6, 3.8))
plt.plot(gpus, ideal, "k--", lw=1.5, label="ideal linear scaling")
plt.plot(gpus, throughput, "o-", lw=2, label="measured")
plt.xlabel("# slow-brain planners (GPUs)"); plt.ylabel("actions planned / s")
plt.title("Fleet throughput scales with the cluster")
plt.legend(); plt.tight_layout(); plt.show()

---
## 7. Closing the loop: distill the slow brain into a fast brain

Search is how you *discover* good actions, but you do not want to pay for it on every step forever. The **"o1 moment for robots"** is to use the expensive planner as a **data engine**: mine `(state → best action)` pairs across the fleet, then **distill** them into a small policy that reproduces the planner's decisions in a single forward pass. The slow brain searches once, and the fast brain inherits the answer and runs on-robot in milliseconds.

This is a well-worn pattern. For example, **AlphaGo to AlphaZero** did exactly this: Monte Carlo Tree Search is the expensive planner, and a policy network is trained to imitate the search's improved move so it can play in one forward pass. In robotics, **Guided Policy Search** uses trajectory optimization as the teacher and distills it into a neural-network controller. And it is how **o1-style** reasoning is compressed into smaller models, by training on the traces an expensive search produced. We apply the same recipe to a world model.

*(NOTE: This cell is a mechanism demo, not a trained product. We mine a small label set from a single clip, so do not read the numbers as a real policy. We still do two honest things: sweep the goal so the planner returns a genuine range of actions, and hold out a validation split so the fast brain has to generalize rather than memorize. Production distillation runs this same loop over millions of diverse states, the workload Ray was built to sustain.)*

In [ ]:
# Slow brain as a data engine: mine (world-model latent -> planned action) labels across all GPUs.
# To get a genuine spread of actions instead of the same answer repeated, we sweep the goal along
# the demo clip: from "stay where you are" to the full target, and back. Nearer goals call for a
# smaller action and farther goals a larger one, so the planner returns a real range of labels.
rng = np.random.default_rng(0)
goal_pose = traj["states"][0, 1]

def observe(start, target, alpha, noise=6):
    """A slightly noisy current frame, plus a goal interpolated alpha-of-the-way to the target."""
    g = (1 - alpha) * start.astype(np.float32) + alpha * target.astype(np.float32)
    g = np.clip(g + rng.integers(-noise, noise, g.shape), 0, 255).astype(np.uint8)
    s = np.clip(start.astype(np.float32) + rng.integers(-noise, noise, start.shape), 0, 255).astype(np.uint8)
    return s, g

sweeps = [(ctx_img, goal_img, pose), (goal_img, ctx_img, goal_pose)]   # toward the goal, and back
specs = [(s, t, p, a) for (s, t, p) in sweeps
                      for a in np.linspace(0.0, 1.0, 10)
                      for _ in range(5)]                               # 100 planning problems

futs = [planners[i % len(planners)].plan_from_images.remote(
            *observe(s, t, a), p, samples=64, cem_steps=4, rollout=1)
        for i, (s, t, p, a) in enumerate(specs)]
labels = ray.get(futs)
X = np.stack([r["feat"] for r in labels])      # mean-pooled context+goal world-model latents
Y = np.stack([r["action"] for r in labels])    # the planner's chosen action for each goal
print(f"mined {len(X)} labels  |  features {X.shape}  ->  actions {Y.shape}")
print(f"action spread (std per dim): {np.round(Y.std(0), 3)}")

In [ ]:
# Train the fast brain: a small MLP from a compact summary of the world-model latent to an action.
# We hold out a validation split and keep the model deliberately small, so the numbers show it
# generalizing rather than memorizing the planner's labels.
import torch, torch.nn as nn

perm = np.random.default_rng(1).permutation(len(X))
ntr = int(0.8 * len(X)); tr, va = perm[:ntr], perm[ntr:]

# Standardize on the training split, then keep only the top principal components (a compact feature).
mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
Xs = (X - mu) / sd
_, _, Vt = np.linalg.svd(Xs[tr], full_matrices=False)
Xt = torch.tensor(Xs @ Vt[:10].T, dtype=torch.float32)   # 2816-dim latent -> 10-dim feature
Yt = torch.tensor(Y, dtype=torch.float32)

# Predicting the mean action is the do-nothing baseline; a useful policy should beat it on held-out data.
baseline = ((Yt[tr].mean(0) - Yt[va]) ** 2).mean().item()

fast_policy = nn.Sequential(nn.Linear(Xt.shape[1], 16), nn.ReLU(), nn.Linear(16, 7))
opt = torch.optim.Adam(fast_policy.parameters(), lr=1e-2, weight_decay=1e-2)
for epoch in range(300):
    opt.zero_grad()
    loss = ((fast_policy(Xt[tr]) - Yt[tr]) ** 2).mean()
    loss.backward(); opt.step()
    if epoch % 100 == 0 or epoch == 299:
        with torch.no_grad():
            val = ((fast_policy(Xt[va]) - Yt[va]) ** 2).mean().item()
        print(f"epoch {epoch:>3}   train MSE = {loss.item():.4f}   val MSE = {val:.4f}")
print(f"(predict-the-mean baseline val MSE = {baseline:.4f})")

In [ ]:
# The payoff: the fast brain reproduces the planner's action in ~milliseconds, no search.
fast_policy.eval()
with torch.no_grad():
    t0 = time.time()
    for _ in range(1000):
        _ = fast_policy(Xt[:1])
    fast_ms = (time.time() - t0) / 1000 * 1e3

print(f"fast brain (distilled policy): {fast_ms:.3f} ms / action  — robot-deployable")
print(f"slow brain (full CEM search):  ~98,000 ms / action  — cloud-only")
print(f"speedup at inference: ~{98_000 / fast_ms:,.0f}x")

---
## 8. Scaling on production hardware

Every number above is from a **4× L4 dev cluster**. This is a modest inference GPU (~24 GB, ~120 bf16 TFLOPS). The L4 is what makes the gap so stark and production accelerators close most of it. The Ray orchestration is **identical**. Thus the curves simply shift down and to the left:

- **Faster GPUs.** H100 / H200 / B200 deliver **~10–30×** the bf16/fp8 throughput of an L4. The ~98 s full-strength plan drops toward **single-digit seconds per action**.
- **More VRAM → bigger search per forward.** 80–192 GB lets you run far larger CEM sample batches in a single forward pass, so *better plans at equal wall-clock*, since CEM samples are just a batch dimension.
- **Data-parallel search.** A single plan's sample batch shards across GPUs with Ray. This is a near-linear speedup *within one action*, on top of the across-action fan-out shown here.
- **Kernel-level wins.** fp8 (Hopper/Blackwell), `torch.compile`, and FlashAttention compound for several more multiples.
- **The fast brain is unaffected** since it already runs in milliseconds and ships to the robot regardless of cloud hardware.

The thesis is hardware-agnostic: **planning is elastic cloud compute, deployment is a cheap distilled policy.** Better hardware buys deeper search or more robots per dollar and not a different architecture.

## 9. The standing architecture

Run this loop **continuously**: the slow brain plans across the fleet's hardest states in the cloud, mines fresh `(state → action)` labels, and distills them into the fast policy, while robots pull periodic policy downloads. The fleet's reflexes keep improving from collective experience. This creates a **flywheel** where compute spent thinking in the cloud becomes capability deployed at the edge.

That is what production-scale Physical AI looks like, and it is a Ray + Anyscale workload end to end where stateful GPU actors, elastic fan-out, shared storage, and a distillation pipeline are orchestrated as one system.

### What we showed

1. A world model (**V-JEPA 2-AC**) the robot can imagine and **search** inside — test-time compute for control.
2. The **compute↔quality curve** (deeper) and **fleet throughput** (wider), both fanned out with Ray.
3. Why search is **cloud-only** (~98 s/action on an L4) and how production hardware shrinks that.
4. **Distillation** of the slow brain into a millisecond fast brain. This is the o1 moment for robots.

**References:** V-JEPA 2 (Meta AI, 2024–2025, `facebookresearch/vjepa2`, MIT) · Cross-Entropy Method for planning · o1-style test-time compute.

In [ ]:
# Release the GPU actors (the §3 planner is included in `planners`).
for p in planners:
    ray.kill(p)
print("planners released.")